In [ ]:
import sys
from pathlib import Path
parent_dir = str(Path().resolve().parent)
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)


from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"


from IPython import get_ipython
from pprint import pformat
formatter = get_ipython().display_formatter.formatters['text/plain']
formatter.for_type(dict, lambda obj, p, _: p.text(pformat(obj, width=120, sort_dicts=False)))
formatter.for_type(list, lambda obj, p, _: p.text(pformat(obj, width=120, sort_dicts=False)))

In [ ]:
import torch
import numpy as np

def seed_everything(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)

seed_everything(42)

In [ ]:
from statistics import mean
from src.evaluate import EvalRecord, MetricResult, EvalResult


def aggregate_eval_results(results: list[EvalResult]) -> EvalResult:
    if not results:
        return EvalResult()

    def _aggregate_metric(metric_name: str) -> MetricResult | None:
        metric_results = [getattr(result, metric_name) for result in results]
        metric_results = [metric for metric in metric_results if metric is not None]
        if not metric_results:
            return None

        details: list[EvalRecord] = [
            record
            for metric in metric_results
            for record in metric.details
        ]
        score = mean(metric.score for metric in metric_results)

        return MetricResult(score=score, details=details)

    return EvalResult(
        success=_aggregate_metric("success"),
        locality=_aggregate_metric("locality"),
        portability=_aggregate_metric("portability"),
    )


def format_step_log(step: int, total_steps: int, result: EvalResult) -> str:
    parts: list[str] = []

    if result.success is not None:
        parts.append(f"success: {result.success.score:.4f} (n={len(result.success.details)})")
    if result.locality is not None:
        parts.append(f"locality: {result.locality.score:.4f} (n={len(result.locality.details)})")
    if result.portability is not None:
        parts.append(f"portability: {result.portability.score:.4f} (n={len(result.portability.details)})")

    metrics_text = " --- ".join(parts) if parts else "no metrics"
    return f"[{step}/{total_steps}] {metrics_text}"

In [ ]:
from dataclasses import asdict
from time import perf_counter

from tqdm.notebook import trange

from src.editors import Editor, EditRequest
from src.evaluate import EvalSuite, evaluate


async def run_benchmark(
    editor: Editor,
    edit_requests: list[EditRequest],
    eval_suites: list[EvalSuite],
    eval_batch_size: int,
    eval_milestones: list[int],
    **kwargs
) -> list[dict]:

    if not eval_milestones:
        raise ValueError("eval_milestones must not be empty")

    eval_milestones = sorted(eval_milestones)
    if eval_milestones[0] <= 0:
        raise ValueError("Milestones must be positive integers")

    max_step = eval_milestones[-1]
    if len(edit_requests) < max_step or len(eval_suites) < max_step:
        raise ValueError(
            f"Need at least {max_step} requests/suites, got {len(edit_requests)} requests and {len(eval_suites)} suites."
        )

    dataset_name = kwargs.get("dataset_name")
    dataset_variant = kwargs.get("dataset_variant")
    editing_method = kwargs.get("editing_method")
    edited_model = kwargs.get("edited_model")
    print(
        f"[run-start] dataset={dataset_name}/{dataset_variant} editing_method={editing_method} edited_model={edited_model} "
        f"milestones={eval_milestones} batch_size={eval_batch_size}"
    )

    run_start = perf_counter()

    milestones = set(eval_milestones)
    milestone_results: list[dict] = []

    for step in trange(1, max_step + 1, desc="Benchmark"):
        editor.edit(edit_requests[step - 1])

        if step not in milestones:
            continue

        eval_results = await evaluate(
            editor.model,
            editor.tokenizer,
            eval_suites[:step],
            batch_size=eval_batch_size,
            **kwargs
        )

        step_result = aggregate_eval_results(eval_results)
        print(format_step_log(step, max_step, step_result))

        milestone_results.append({
            "step": step,
            "edits": [
                {
                    "prompt": req.prompt,
                    "old_ground_truth": req.old_ground_truth,
                    "new_ground_truth": req.new_ground_truth,
                }
                for req in edit_requests[:step]
            ],
            "result": asdict(step_result)
        })

    elapsed = perf_counter() - run_start
    print(f"[run-end] dataset={dataset_name}/{dataset_variant} elapsed={elapsed:.2f}s ({elapsed / 60:.2f} min)")
    return milestone_results

In [ ]:
dataset_variant = ["cloze", "qna"][0]
model_name = ["qwen2.5-7b"][0]
editing_methods = ["ft", "rome", "memit", "grace", "wise", "alpha_edit"]

eval_batch_size = 16
eval_milestones = [1, 5, 10, 50, 100]
n_records = eval_milestones[-1]

model_generation = {
    "max_new_tokens": 64,
    # "do_sample": True,
    # "temperature": 0.7,
    # "top_p": 0.8,
    # "top_k": 20,
}

llm_judge = {
    # "base_url": "http://localhost:8080/v1",
    # "api_key": "local",
    "judge_model": "openai/gpt-oss-20b",
    # "judge_model": "Qwen/Qwen3-32B-AWQ",
    "judge_extra_config": {
        "temperature": 0.1,
        "extra_body": {
            "reasoning_effort": "medium"
        }
        # "presence_penalty": 1.5,
        # "extra_body": {
        #     "chat_template_kwargs": {"enable_thinking": True}
        # }
    },
    "max_concurrency": 4,
    "max_retries": 5,
    # "init_retry_delay": 1.0,
    # "max_retry_delay": 30.0,
}

In [ ]:
import random
from datasets import load_from_disk

from src.editors import load_editor
from src.evaluate import EvalPair, save_benchmark_results


GLOBAL_MODEL = None
GLOBAL_TOKENIZER = None

cfact = load_from_disk("../data/datasets/cfact")
mquake = load_from_disk("../data/datasets/mquake")

In [ ]:
def build_cfact_inputs(dataset, n_edits: int) -> tuple[list[EditRequest], list[EvalSuite]]:
    def build_pair(record):
        edit_request = EditRequest(
            record['subject'],
            record['prompt'],
            record['old_ans'],
            record['new_ans']
        )
        eval_suite = EvalSuite(
            success=[EvalPair(record['prompt'], record['new_ans'])] +\
                    [EvalPair(prompt, record['new_ans']) for prompt in record['rephrases']],
            locality=[EvalPair(pair['prompt'], pair['ans']) for pair in record['locality']]
        )
        return edit_request, eval_suite

    if n_edits <= 0:
        raise ValueError("n must be > 0")
    if n_edits > len(dataset):
        raise ValueError(f"Requested {n_edits} edits, but only {len(dataset)} edits are available.")

    selected = list(dataset)
    random.shuffle(selected)
    selected = selected[:n_edits]

    pairs = [build_pair(record) for record in selected]
    edit_requests = [request for request, _ in pairs]
    eval_suites = [suite for _, suite in pairs]
    return edit_requests, eval_suites


cfact_requests, cfact_suites = build_cfact_inputs(cfact[dataset_variant], n_records)


for editing_method in editing_methods:
    editor = load_editor(
        f"../src/config/{editing_method}/{model_name}.yaml",
        model=GLOBAL_MODEL,
        tokenizer=GLOBAL_TOKENIZER,
    )
    try:
        if GLOBAL_MODEL is None or GLOBAL_TOKENIZER is None:
            GLOBAL_MODEL = editor.model
            GLOBAL_TOKENIZER = editor.tokenizer

        cfact_milestone_results = await run_benchmark(
            editor=editor,
            edit_requests=cfact_requests,
            eval_suites=cfact_suites,
            eval_batch_size=eval_batch_size,
            eval_milestones=eval_milestones,
            dataset_name="cfact",
            dataset_variant=dataset_variant,
            editing_method=editing_method,
            edited_model=model_name,
            model_generation_config=model_generation,
            llm_judge_config=llm_judge,
        )

        save_benchmark_results(
            results=cfact_milestone_results,
            dataset_variant=dataset_variant,
            edited_model=model_name,
            editing_method=editing_method,
            milestones=eval_milestones,
            dataset_name="cfact",
        )

        editor.reset()
    finally:
        editor.reset()
        del editor

In [ ]:
def build_mquake_inputs(dataset, n_edits: int) -> tuple[list[EditRequest], list[EvalSuite]]:
    def build_pair(record):
        edit_request = EditRequest(
            record['subject'],
            record['prompt'],
            record['old_ans'],
            record['new_ans']
        )
        eval_suite = EvalSuite(
            portability=[EvalPair(prompt, record['portability']['new_ans']) for prompt in record['portability']['prompts']]
        )
        return edit_request, eval_suite

    if n_edits <= 0:
        raise ValueError("n must be > 0")
    if n_edits > len(dataset):
        raise ValueError(f"Requested {n_edits} edits, but only {len(dataset)} edits are available.")

    selected = list(dataset)
    random.shuffle(selected)
    selected = selected[:n_edits]

    pairs = [build_pair(record) for record in selected]
    edit_requests = [request for request, _ in pairs]
    eval_suites = [suite for _, suite in pairs]
    return edit_requests, eval_suites


mquake_requests, mquake_suites = build_mquake_inputs(mquake[dataset_variant], n_records)


for editing_method in editing_methods:
    editor = load_editor(
        f"../src/config/{editing_method}/{model_name}.yaml",
        model=GLOBAL_MODEL,
        tokenizer=GLOBAL_TOKENIZER,
    )
    try:
        if GLOBAL_MODEL is None or GLOBAL_TOKENIZER is None:
            GLOBAL_MODEL = editor.model
            GLOBAL_TOKENIZER = editor.tokenizer

        mquake_milestone_results = await run_benchmark(
            editor=editor,
            edit_requests=mquake_requests,
            eval_suites=mquake_suites,
            eval_batch_size=eval_batch_size,
            eval_milestones=eval_milestones,
            dataset_name="mquake",
            dataset_variant=dataset_variant,
            editing_method=editing_method,
            edited_model=model_name,
            model_generation_config=model_generation,
            llm_judge_config=llm_judge,
        )

        save_benchmark_results(
            results=mquake_milestone_results,
            dataset_variant=dataset_variant,
            edited_model=model_name,
            editing_method=editing_method,
            milestones=eval_milestones,
            dataset_name="mquake",
        )

        editor.reset()
    finally:
        editor.reset()
        del editor